# Feature-duplicate, test-count, FDR, stability, and AUC audit

## tl;dr

- The 9,450-column HCFIM matrix contains **4,800 exact structural aliases (50.8%)** and **4,650 unique value vectors**. H contributes 2,100 aliases across distance labels and FIM contributes 2,700 across distance and entropy-family labels; C has no exact aliases.
- The historical **875-test** family collapses distance labels but still counts seven transform-labeled FIM coordinates. Collapsing FIM by its three ordinal probability domains (`m=3,4,5`) gives the defensible **775-test** family.
- Benjamini-Hochberg discoveries changed from **5 to 6 for control versus AD** and remained **0 for control versus FTD** when moving from 875 to 775 tests. Applying BH to all 1,575 raw mean-coordinate labels produced 18 AD rows because exact aliases were counted repeatedly; those are not 18 independent findings.
- In 10 repeated five-fold comparisons at matched `k=30`, deduplication changed PSD+HCFIM mean ROC AUC by **+0.004 for AD** and **+0.015 for FTD**. Mean pairwise selection Jaccard changed by **+0.067** and **+0.050**, respectively.
- The original single-partition pipeline was exactly reproduced before comparison. Its deduplicated AUC changed by **-0.029 for AD** and **-0.025 for FTD**, illustrating the instability of one partition; the repeated paired analysis is the more reliable sensitivity estimate.

## Context & Methods

This diagnostic notebook audits the 9,450-column HCFIM inventory, reconstructs the historical 875-test and corrected 775-test multiplicity families, verifies exact structural aliases, and quantifies their impact on FDR, feature-selection stability, and subject-level ROC AUC. It uses the released subject-level feature matrix and the same partitions and estimators as the confirmatory and repeated-primary notebooks; no EEG epochs are recomputed.

### Key assumptions

- H is invariant to the probability-distance choice for a fixed entropy transform and probability domain.
- FIM depends on the probability vector and state ordering, not on the entropy functional or probability-distance label. Ordinal Shannon, Rényi, and Tsallis variants with the same `m` and `tau` therefore share FIM.
- C remains transform- and distance-specific.
- Only structural aliases are removed from classification. Accidental empirical equality between scientifically distinct coordinates is reported but not used for deduplication.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import os
import platform
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from scipy import stats
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 17536
N_REPEATS = 10
OUTER_FOLDS = 5
K_VALUES = [21, 30]
N_TREES_REPEATED = 200
MAX_DEPTH_REPEATED = 3

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS = PROJECT_ROOT / "results"
INPUT_FEATURES = RESULTS / "subject_features_full.csv"
ORIGINAL_REPEATED = RESULTS / "repeated_primary_validation"
ORIGINAL_CONFIRMATORY = RESULTS / "confirmatory_sensitivity"
OUT = RESULTS / "feature_duplicate_audit"
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
})

COLORS = {"original": "#315B7D", "deduplicated": "#D18B32", "delta": "#6F8F3D"}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Outputs: {OUT}")


## Data

### 1. Load and validate the participant-level matrix

In [ ]:
subject_features = pd.read_csv(INPUT_FEATURES)
assert len(subject_features) == 88
assert subject_features["participant_id"].is_unique

psd_columns = [column for column in subject_features if column.startswith("psd__")]
hcfim_columns = [column for column in subject_features if column.startswith("hcfim__")]
assert len(psd_columns) == 210
assert len(hcfim_columns) == 9450
assert np.isfinite(subject_features[psd_columns + hcfim_columns].to_numpy(float)).all()

input_hash = sha256_file(INPUT_FEATURES)
manifest_path = ORIGINAL_REPEATED / "notebook_10_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
recorded_hashes = list(manifest.get("input_files", {}).values())
assert input_hash in recorded_hashes, "The repeated-primary outputs do not match the current subject feature matrix."

display(pd.DataFrame({
    "item": ["Participants", "PSD columns", "HCFIM columns", "Input SHA-256 matches notebook 10"],
    "value": [len(subject_features), len(psd_columns), len(hcfim_columns), True],
}))


### 2. Parse the feature grid and define structural probability domains

In [ ]:
COLUMN_PATTERN = re.compile(
    r"^hcfim__(?P<transform>.+?)__(?P<distance>.+?)__(?P<region>.+?)__"
    r"(?P<band>.+?)__(?P<coordinate>H|C|FIM)__(?P<stat>mean|std|median|p25|p75|iqr)$"
)

def parse_hcfim_column(column: str) -> dict:
    match = COLUMN_PATTERN.match(column)
    if not match:
        raise ValueError(f"Unparsed HCFIM column: {column}")
    return {"column": column, **match.groupdict()}

def ordinal_probability_domain(transform: str) -> str:
    match = re.search(r"_m(\d+)_t(\d+)$", transform)
    if transform.startswith("ordinal_") and match:
        return f"ordinal_m{match.group(1)}_t{match.group(2)}"
    return transform

def structural_key_from_parts(row) -> tuple:
    transform = row["transform"]
    distance = row["distance"]
    coordinate = row["coordinate"]
    if coordinate == "H":
        distance = "distance_invariant"
    elif coordinate == "FIM":
        distance = "distance_invariant"
        transform = ordinal_probability_domain(transform)
    return (transform, distance, row["region"], row["band"], coordinate, row["stat"])

def legacy_875_key_from_parts(row) -> tuple:
    transform = row["transform"]
    distance = row["distance"]
    coordinate = row["coordinate"]
    if coordinate in {"H", "FIM"}:
        distance = "distance_invariant"
    return (transform, distance, row["region"], row["band"], coordinate, row["stat"])

def canonical_feature_key(column: str) -> str:
    if not column.startswith("hcfim__"):
        return column
    parts = parse_hcfim_column(column)
    return "::".join(structural_key_from_parts(parts))

metadata = pd.DataFrame([parse_hcfim_column(column) for column in hcfim_columns])
metadata["probability_domain"] = metadata["transform"].map(ordinal_probability_domain)
metadata["structural_key"] = metadata.apply(structural_key_from_parts, axis=1)
metadata["legacy_875_key"] = metadata.apply(legacy_875_key_from_parts, axis=1)

grid_counts = {field: metadata[field].nunique() for field in ["region", "band", "transform", "distance", "coordinate", "stat"]}
assert grid_counts == {"region": 5, "band": 5, "transform": 7, "distance": 3, "coordinate": 3, "stat": 6}
assert np.prod(list(grid_counts.values())) == 9450
display(pd.Series(grid_counts, name="levels").to_frame())


## Results

### 3. Reconstruct 9,450, 875, and 775 and verify exact aliases

In [ ]:
def canonical_priority(column: str) -> tuple:
    parts = parse_hcfim_column(column)
    transform_priority = 0 if parts["transform"].startswith("ordinal_shannon") else 1
    distance_priority = {"euclidean": 0, "jensen_shannon": 1, "wootters": 2}[parts["distance"]]
    return (transform_priority, distance_priority, column)

structural_groups = metadata.groupby("structural_key", sort=False)["column"].apply(list)
legacy_groups = metadata.groupby("legacy_875_key", sort=False)["column"].apply(list)
assert len(structural_groups) == 4650
assert len(legacy_groups) == 5250  # 875 mean-coordinate tests x 6 summaries

structural_audit_rows = []
for key, columns in structural_groups.items():
    representative = min(columns, key=canonical_priority)
    reference = subject_features[representative].to_numpy(float)
    exact = all(np.array_equal(reference, subject_features[column].to_numpy(float)) for column in columns)
    max_abs_difference = max(
        float(np.max(np.abs(reference - subject_features[column].to_numpy(float)))) for column in columns
    )
    structural_audit_rows.append({
        "structural_key": "::".join(key),
        "coordinate": key[4],
        "group_size": len(columns),
        "representative": representative,
        "all_exact": exact,
        "max_abs_difference": max_abs_difference,
    })

structural_audit = pd.DataFrame(structural_audit_rows)
assert structural_audit["all_exact"].all()
assert structural_audit["max_abs_difference"].eq(0).all()

representative_by_key = structural_audit.set_index("structural_key")["representative"].to_dict()
metadata["structural_key_text"] = metadata["structural_key"].map(lambda key: "::".join(key))
metadata["representative"] = metadata["structural_key_text"].map(representative_by_key)
metadata["is_structural_representative"] = metadata["column"].eq(metadata["representative"])
metadata["structural_group_size"] = metadata["structural_key_text"].map(structural_audit.set_index("structural_key")["group_size"])

def exact_vector_hash(column: str) -> str:
    values = np.ascontiguousarray(subject_features[column].to_numpy(dtype="<f8"))
    return hashlib.sha256(values.tobytes()).hexdigest()

metadata["exact_vector_hash"] = [exact_vector_hash(column) for column in metadata["column"]]
exact_group_sizes = metadata.groupby("exact_vector_hash")["column"].size()
metadata["exact_group_size"] = metadata["exact_vector_hash"].map(exact_group_sizes)
empirical_unique_vectors = metadata["exact_vector_hash"].nunique()

deduplicated_hcfim_columns = sorted(structural_audit["representative"].tolist())
assert len(deduplicated_hcfim_columns) == 4650

count_reconstruction = pd.DataFrame([
    {"inventory": "Raw HCFIM feature matrix", "count": 9450, "formula": "25 region-band x 7 transforms x 3 distances x 3 coordinates x 6 summaries"},
    {"inventory": "Structural unique HCFIM features", "count": 4650, "formula": "25 x 6 x (7 H + 21 C + 3 probability-domain FIM)"},
    {"inventory": "Redundant structural aliases", "count": 4800, "formula": "9450 - 4650"},
    {"inventory": "Raw mean-coordinate tests", "count": 1575, "formula": "25 x 7 x 3 x 3"},
    {"inventory": "Historical partially deduplicated tests", "count": 875, "formula": "25 x (7 H + 21 C + 7 transform-labeled FIM)"},
    {"inventory": "Correct probability-domain tests", "count": 775, "formula": "25 x (7 H + 21 C + 3 probability-domain FIM)"},
])

coordinate_inventory = pd.DataFrame([
    {
        "coordinate": coordinate,
        "raw_columns": int((metadata["coordinate"] == coordinate).sum()),
        "structural_unique": int(structural_audit.loc[structural_audit["coordinate"] == coordinate].shape[0]),
    }
    for coordinate in ["H", "C", "FIM"]
])
coordinate_inventory["redundant_aliases"] = coordinate_inventory["raw_columns"] - coordinate_inventory["structural_unique"]

count_reconstruction.to_csv(OUT / "count_reconstruction.csv", index=False)
coordinate_inventory.to_csv(OUT / "coordinate_duplicate_inventory.csv", index=False)
structural_audit.to_csv(OUT / "structural_duplicate_group_audit.csv", index=False)
metadata.drop(columns=["structural_key", "legacy_875_key"]).to_csv(OUT / "feature_duplicate_inventory.csv", index=False)

display(count_reconstruction)
display(coordinate_inventory)
print(f"Empirically unique 88-participant value vectors: {empirical_unique_vectors:,}")
print(f"Structural groups verified exactly: {len(structural_audit):,}/{len(structural_audit):,}")


### 4. Recalculate FDR under the raw, historical-875, and corrected-775 families

In [ ]:
COMPARISONS = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order] * len(p_values) / np.arange(1, len(p_values) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted = np.empty_like(ranked)
    adjusted[order] = np.clip(ranked, 0, 1)
    return adjusted

def hedges_g(a: np.ndarray, b: np.ndarray) -> float:
    pooled_num = (len(a) - 1) * np.var(a, ddof=1) + (len(b) - 1) * np.var(b, ddof=1)
    pooled = np.sqrt(pooled_num / (len(a) + len(b) - 2))
    if not np.isfinite(pooled) or pooled == 0:
        return np.nan
    d = (np.mean(b) - np.mean(a)) / pooled
    return (1 - 3 / (4 * (len(a) + len(b)) - 9)) * d

mean_metadata = metadata.loc[metadata["stat"] == "mean"].copy()
assert len(mean_metadata) == 1575

def representatives_for_scheme(scheme: str) -> list[str]:
    if scheme == "raw_1575":
        return sorted(mean_metadata["column"])
    key_column = "legacy_875_key" if scheme == "historical_875" else "structural_key"
    groups = mean_metadata.groupby(key_column, sort=False)["column"].apply(list)
    return sorted(min(columns, key=canonical_priority) for columns in groups)

scheme_columns = {
    "raw_1575": representatives_for_scheme("raw_1575"),
    "historical_875": representatives_for_scheme("historical_875"),
    "corrected_775": representatives_for_scheme("corrected_775"),
}
assert {name: len(columns) for name, columns in scheme_columns.items()} == {
    "raw_1575": 1575, "historical_875": 875, "corrected_775": 775,
}

fdr_rows = []
for scheme, columns in scheme_columns.items():
    for comparison, groups in COMPARISONS.items():
        subset = subject_features.loc[subject_features["diagnosis"].isin(groups)]
        start_index = len(fdr_rows)
        for column in columns:
            a = subset.loc[subset["diagnosis"] == groups[0], column].to_numpy(float)
            b = subset.loc[subset["diagnosis"] == groups[1], column].to_numpy(float)
            result = stats.mannwhitneyu(a, b, alternative="two-sided")
            parts = parse_hcfim_column(column)
            fdr_rows.append({
                "scheme": scheme,
                "comparison": comparison,
                "column": column,
                **{key: parts[key] for key in ["transform", "distance", "region", "band", "coordinate"]},
                "probability_domain": ordinal_probability_domain(parts["transform"]),
                "n_control": len(a),
                "n_pathology": len(b),
                "control_mean": np.mean(a),
                "pathology_mean": np.mean(b),
                "hedges_g": hedges_g(a, b),
                "u_statistic": result.statistic,
                "p_value": result.pvalue,
            })
        stop_index = len(fdr_rows)
        q_values = benjamini_hochberg([row["p_value"] for row in fdr_rows[start_index:stop_index]])
        for row, q_value in zip(fdr_rows[start_index:stop_index], q_values):
            row["q_value"] = q_value
            row["significant_fdr_05"] = q_value < 0.05

fdr_results = pd.DataFrame(fdr_rows)
fdr_summary = (
    fdr_results.groupby(["scheme", "comparison"], as_index=False)
    .agg(
        tests=("column", "size"),
        discoveries_fdr_05=("significant_fdr_05", "sum"),
        minimum_p=("p_value", "min"),
        minimum_q=("q_value", "min"),
    )
)
corrected_discoveries = fdr_results.loc[
    (fdr_results["scheme"] == "corrected_775") & fdr_results["significant_fdr_05"]
].sort_values(["comparison", "q_value"])

# Reconcile the corrected implementation with notebook 06.
notebook06_fdr = pd.read_csv(ORIGINAL_CONFIRMATORY / "coordinate_tests_fdr.csv")
corrected_all = fdr_results.loc[fdr_results["scheme"] == "corrected_775"]
reconciled = corrected_all.merge(
    notebook06_fdr[["comparison", "column", "p_value", "q_value"]],
    on=["comparison", "column"], how="outer", suffixes=("_audit", "_nb06"), indicator=True,
)
assert reconciled["_merge"].eq("both").all()
assert np.allclose(reconciled["p_value_audit"], reconciled["p_value_nb06"], rtol=0, atol=1e-15)
assert np.allclose(reconciled["q_value_audit"], reconciled["q_value_nb06"], rtol=0, atol=1e-15)

fdr_results.to_csv(OUT / "fdr_results_all_counting_schemes.csv", index=False)
fdr_summary.to_csv(OUT / "fdr_counting_scheme_summary.csv", index=False)
corrected_discoveries.to_csv(OUT / "corrected_775_fdr_discoveries.csv", index=False)
display(fdr_summary.style.format({"minimum_p": "{:.3g}", "minimum_q": "{:.3g}"}))
display(corrected_discoveries[["comparison", "column", "hedges_g", "p_value", "q_value"]].style.format(precision=4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), sharey=False)
for ax, comparison in zip(axes, COMPARISONS):
    plot_data = fdr_summary.loc[fdr_summary["comparison"] == comparison].copy()
    order = ["raw_1575", "historical_875", "corrected_775"]
    plot_data["scheme"] = pd.Categorical(plot_data["scheme"], categories=order, ordered=True)
    plot_data = plot_data.sort_values("scheme")
    bars = ax.bar(plot_data["scheme"].astype(str), plot_data["discoveries_fdr_05"], color=["#A8B8C8", "#6F8F3D", "#315B7D"])
    y_max = max(1.0, float(plot_data["discoveries_fdr_05"].max()) * 1.15)
    ax.set_ylim(0, y_max)
    for bar, value in zip(bars, plot_data["discoveries_fdr_05"]):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.025 * y_max, f"{int(value)}", ha="center")
    ax.set_title(comparison.replace("_", " ").title())
    ax.set_ylabel("BH discoveries at q < 0.05")
    ax.set_xlabel("Multiplicity family")
    ax.tick_params(axis="x", rotation=20)
fig.suptitle("FDR discoveries depend on whether exact aliases are counted as separate tests")
fig.tight_layout()
fig.savefig(FIGURES / "fdr_counting_scheme_comparison.png", bbox_inches="tight", facecolor="white")
plt.show()


### 5. Re-evaluate the reported single-partition AUC and selection stability

In [ ]:
def pairwise_jaccard(sets: list[set[str]]) -> float:
    values = [len(left & right) / len(left | right) for left, right in itertools.combinations(sets, 2)]
    return float(np.mean(values))

def make_nested_pipeline(columns: list[str], seed: int) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=min(30, len(columns)))),
        ("model", ExtraTreesClassifier(
            n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1,
        )),
    ])

def single_partition_nested(task: str, groups: list[str], columns: list[str], variant: str):
    # Preserve the source row order used by notebook 06; sorting would change
    # the sample indices assigned to the seeded StratifiedKFold partitions.
    frame = subject_features.loc[subject_features["diagnosis"].isin(groups)].reset_index(drop=True)
    X = frame[columns]
    y = (frame["diagnosis"] == groups[1]).astype(int).to_numpy()
    probabilities = np.full(len(y), np.nan)
    predictions = np.full(len(y), -1)
    selections = []
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
        pipeline = make_nested_pipeline(columns, SEED)
        inner = StratifiedKFold(3, shuffle=True, random_state=SEED + fold)
        search = GridSearchCV(
            pipeline, {"model__max_depth": [2, 3, None]}, scoring="roc_auc", cv=inner, n_jobs=1,
        )
        search.fit(X.iloc[train_idx], y[train_idx])
        best = search.best_estimator_
        probabilities[test_idx] = best.predict_proba(X.iloc[test_idx])[:, 1]
        predictions[test_idx] = (probabilities[test_idx] >= 0.50).astype(int)
        support = best.named_steps["select"].get_support()
        selections.append(set(np.asarray(columns)[support]))
    raw_jaccard = pairwise_jaccard(selections)
    canonical_sets = [{canonical_feature_key(column) for column in selected} for selected in selections]
    canonical_jaccard = pairwise_jaccard(canonical_sets)
    unique_slots = [len(selected) for selected in canonical_sets]
    return {
        "task": task,
        "variant": variant,
        "candidate_features": len(columns),
        "roc_auc": roc_auc_score(y, probabilities),
        "pr_auc": average_precision_score(y, probabilities),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "f1": f1_score(y, predictions),
        "raw_name_jaccard": raw_jaccard,
        "canonical_domain_jaccard": canonical_jaccard,
        "mean_unique_selected_domains": float(np.mean(unique_slots)),
        "min_unique_selected_domains": int(np.min(unique_slots)),
    }, selections

tasks = COMPARISONS
# Notebook 06 retained the source-matrix column order (HCFIM before PSD).
# Preserve that order because tied SelectKBest scores are resolved by position.
full_eeg_columns = [
    column for column in subject_features
    if column.startswith("psd__") or column.startswith("hcfim__")
]
deduplicated_hcfim_set = set(deduplicated_hcfim_columns)
deduplicated_eeg_columns = [
    column for column in subject_features
    if column.startswith("psd__")
    or (column.startswith("hcfim__") and column in deduplicated_hcfim_set)
]

single_rows = []
single_selection_rows = []
single_start = time.time()
for task, groups in tasks.items():
    for variant, columns in [("original", full_eeg_columns), ("deduplicated", deduplicated_eeg_columns)]:
        print(f"Single-partition nested audit: {task} | {variant} | {len(columns)} candidates")
        metrics, selections = single_partition_nested(task, groups, columns, variant)
        single_rows.append(metrics)
        for fold, selected in enumerate(selections, start=1):
            for feature in selected:
                single_selection_rows.append({"task": task, "variant": variant, "fold": fold, "feature": feature})

single_comparison = pd.DataFrame(single_rows)
single_selections = pd.DataFrame(single_selection_rows)
single_comparison.to_csv(OUT / "reported_single_partition_comparison.csv", index=False)
single_selections.to_csv(OUT / "reported_single_partition_selected_features.csv", index=False)
print(f"Single-partition audit completed in {(time.time() - single_start) / 60:.1f} min")

reported_auc = pd.read_csv(ORIGINAL_CONFIRMATORY / "covariate_model_comparison.csv")
reported_auc = reported_auc.loc[reported_auc["label"] == "EEG", ["comparison", "roc_auc"]]
reported_auc = reported_auc.rename(columns={"comparison": "task", "roc_auc": "reported_roc_auc"})
reported_stability = pd.read_csv(ORIGINAL_CONFIRMATORY / "feature_selection_stability.csv").rename(columns={"comparison": "task"})
original_recomputed = single_comparison.loc[single_comparison["variant"] == "original"]
check = original_recomputed.merge(reported_auc, on="task").merge(reported_stability, on="task")
assert np.allclose(check["roc_auc"], check["reported_roc_auc"], rtol=0, atol=1e-12)
assert np.allclose(check["raw_name_jaccard"], check["mean_pairwise_jaccard"], rtol=0, atol=1e-12)

single_delta = single_comparison.pivot(index="task", columns="variant", values=["roc_auc", "raw_name_jaccard", "canonical_domain_jaccard"])
single_delta.columns = [f"{metric}_{variant}" for metric, variant in single_delta.columns]
single_delta = single_delta.reset_index()
single_delta["auc_delta_dedup_minus_original"] = single_delta["roc_auc_deduplicated"] - single_delta["roc_auc_original"]
single_delta["raw_jaccard_delta"] = single_delta["raw_name_jaccard_deduplicated"] - single_delta["raw_name_jaccard_original"]
single_delta.to_csv(OUT / "reported_single_partition_delta_summary.csv", index=False)
display(single_delta.style.format(precision=3))


### 6. Repeat matched-k validation after structural deduplication

In [ ]:
original_repeat_metrics = pd.read_csv(ORIGINAL_REPEATED / "repeated_primary_metrics.csv")
original_selected_features = pd.read_csv(ORIGINAL_REPEATED / "repeated_primary_selected_features.csv")

audit_feature_sets = {
    "HCFIM": deduplicated_hcfim_columns,
    "PSD+HCFIM": psd_columns + deduplicated_hcfim_columns,
}

def make_repeated_pipeline(k: int, seed: int) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("model", ExtraTreesClassifier(
            n_estimators=N_TREES_REPEATED,
            max_depth=MAX_DEPTH_REPEATED,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        )),
    ])

def repeated_deduplicated_validation(task: str, groups: list[str], feature_set: str, k: int):
    frame = subject_features.loc[subject_features["diagnosis"].isin(groups)].sort_values("participant_id").reset_index(drop=True)
    columns = audit_feature_sets[feature_set]
    X = frame[columns].to_numpy(float)
    y = (frame["diagnosis"] == groups[1]).astype(int).to_numpy()
    repeat_rows, selection_rows = [], []
    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        probabilities = np.full(len(y), np.nan)
        predictions = np.full(len(y), -1)
        for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            fit_seed = SEED + repeat * 1000 + fold
            estimator = make_repeated_pipeline(k, fit_seed)
            estimator.fit(X[train_idx], y[train_idx])
            probabilities[test_idx] = estimator.predict_proba(X[test_idx])[:, 1]
            predictions[test_idx] = (probabilities[test_idx] >= 0.50).astype(int)
            support = estimator.named_steps["select"].get_support()
            for feature in np.asarray(columns)[support]:
                selection_rows.append({
                    "task": task, "feature_set": feature_set, "k": k,
                    "repeat": repeat, "fold": fold, "feature": feature,
                })
        repeat_rows.append({
            "task": task, "feature_set": feature_set, "k": k, "repeat": repeat,
            "roc_auc": roc_auc_score(y, probabilities),
            "pr_auc": average_precision_score(y, probabilities),
            "balanced_accuracy": balanced_accuracy_score(y, predictions),
            "f1": f1_score(y, predictions),
        })
    return pd.DataFrame(repeat_rows), pd.DataFrame(selection_rows)

repeat_outputs, selection_outputs = [], []
repeat_start = time.time()
for task, groups in tasks.items():
    for feature_set in audit_feature_sets:
        for k in K_VALUES:
            print(f"Repeated audit: {task} | {feature_set} | k={k}")
            metrics, selections = repeated_deduplicated_validation(task, groups, feature_set, k)
            repeat_outputs.append(metrics)
            selection_outputs.append(selections)

dedup_repeat_metrics = pd.concat(repeat_outputs, ignore_index=True)
dedup_selected_features = pd.concat(selection_outputs, ignore_index=True)
dedup_repeat_metrics.to_csv(OUT / "deduplicated_repeated_metrics.csv", index=False)
dedup_selected_features.to_csv(OUT / "deduplicated_repeated_selected_features.csv", index=False)
print(f"Repeated deduplicated validation completed in {(time.time() - repeat_start) / 60:.1f} min")

original_relevant_metrics = original_repeat_metrics.loc[
    original_repeat_metrics["feature_set"].isin(audit_feature_sets)
].copy()
paired_auc = original_relevant_metrics.merge(
    dedup_repeat_metrics,
    on=["task", "feature_set", "k", "repeat"],
    suffixes=("_original", "_deduplicated"),
    validate="one_to_one",
)
paired_auc["auc_delta"] = paired_auc["roc_auc_deduplicated"] - paired_auc["roc_auc_original"]
paired_auc.to_csv(OUT / "paired_auc_by_repeat.csv", index=False)

auc_summary = (
    paired_auc.groupby(["task", "feature_set", "k"], as_index=False)
    .agg(
        original_auc_mean=("roc_auc_original", "mean"),
        deduplicated_auc_mean=("roc_auc_deduplicated", "mean"),
        auc_delta_mean=("auc_delta", "mean"),
        auc_delta_sd=("auc_delta", "std"),
        auc_delta_p2_5=("auc_delta", lambda values: np.percentile(values, 2.5)),
        auc_delta_p97_5=("auc_delta", lambda values: np.percentile(values, 97.5)),
    )
)
auc_summary.to_csv(OUT / "paired_auc_summary.csv", index=False)
display(auc_summary.style.format(precision=3))


### 7. Compare raw-name and probability-domain selection stability

In [ ]:
def stability_from_long(group: pd.DataFrame, canonicalize: bool) -> tuple[float, float, float]:
    fold_sets = []
    unique_counts = []
    for _, fold in group.groupby(["repeat", "fold"]):
        selected = set(fold["feature"])
        if canonicalize:
            selected = {canonical_feature_key(feature) for feature in selected}
        fold_sets.append(selected)
        unique_counts.append(len(selected))
    pairwise = [len(left & right) / len(left | right) for left, right in itertools.combinations(fold_sets, 2)]
    return float(np.mean(pairwise)), float(np.median(pairwise)), float(np.mean(unique_counts))

stability_rows = []
for keys, original_group in original_selected_features.loc[
    original_selected_features["feature_set"].isin(audit_feature_sets)
].groupby(["task", "feature_set", "k"]):
    task, feature_set, k = keys
    dedup_group = dedup_selected_features.loc[
        (dedup_selected_features["task"] == task)
        & (dedup_selected_features["feature_set"] == feature_set)
        & (dedup_selected_features["k"] == k)
    ]
    original_raw = stability_from_long(original_group, canonicalize=False)
    original_canonical = stability_from_long(original_group, canonicalize=True)
    dedup_raw = stability_from_long(dedup_group, canonicalize=False)
    stability_rows.append({
        "task": task, "feature_set": feature_set, "k": k,
        "original_raw_jaccard": original_raw[0],
        "original_canonical_jaccard": original_canonical[0],
        "deduplicated_jaccard": dedup_raw[0],
        "jaccard_delta_dedup_minus_original_raw": dedup_raw[0] - original_raw[0],
        "original_mean_unique_domains_selected": original_canonical[2],
        "deduplicated_mean_unique_domains_selected": dedup_raw[2],
    })

stability_summary = pd.DataFrame(stability_rows)
stability_summary.to_csv(OUT / "selection_stability_comparison.csv", index=False)
display(stability_summary.style.format(precision=3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.0))

plot_auc = auc_summary.loc[auc_summary["k"] == 30].copy()
task_labels = {"control_vs_alzheimer": "AD", "control_vs_frontotemporal_dementia": "FTD"}
labels = [f"{task_labels[row.task]}\n{row.feature_set}" for row in plot_auc.itertuples()]
x = np.arange(len(plot_auc))
width = 0.36
axes[0].bar(x - width/2, plot_auc["original_auc_mean"], width, label="Original", color=COLORS["original"])
axes[0].bar(x + width/2, plot_auc["deduplicated_auc_mean"], width, label="Deduplicated", color=COLORS["deduplicated"])
axes[0].set_ylim(0.5, max(0.86, plot_auc[["original_auc_mean", "deduplicated_auc_mean"]].to_numpy().max() + 0.04))
axes[0].set_xticks(x, labels)
axes[0].set_ylabel("Mean repeated ROC AUC")
axes[0].set_title("AUC after structural deduplication (k=30)")
axes[0].legend(frameon=False)
for index, row in enumerate(plot_auc.itertuples()):
    axes[0].text(index, max(row.original_auc_mean, row.deduplicated_auc_mean) + 0.01, f"Δ {row.auc_delta_mean:+.3f}", ha="center", fontsize=9)

plot_stability = stability_summary.loc[stability_summary["k"] == 30].copy()
labels_stability = [f"{task_labels[row.task]}\n{row.feature_set}" for row in plot_stability.itertuples()]
x2 = np.arange(len(plot_stability))
axes[1].bar(x2 - width/2, plot_stability["original_raw_jaccard"], width, label="Original raw names", color=COLORS["original"])
axes[1].bar(x2 + width/2, plot_stability["deduplicated_jaccard"], width, label="Deduplicated", color=COLORS["deduplicated"])
axes[1].set_xticks(x2, labels_stability)
axes[1].set_ylabel("Mean pairwise Jaccard across 50 selections")
axes[1].set_title("Selection stability after deduplication (k=30)")
axes[1].legend(frameon=False)

fig.tight_layout()
fig.savefig(FIGURES / "auc_and_selection_stability_deduplication.png", bbox_inches="tight", facecolor="white")
plt.show()


## Takeaways

In [ ]:
ad_fdr = fdr_summary.loc[
    (fdr_summary["comparison"] == "control_vs_alzheimer")
].set_index("scheme")["discoveries_fdr_05"]
ftd_fdr = fdr_summary.loc[
    (fdr_summary["comparison"] == "control_vs_frontotemporal_dementia")
].set_index("scheme")["discoveries_fdr_05"]

primary_auc = auc_summary.loc[auc_summary["k"] == 30]
max_abs_auc_delta = float(primary_auc["auc_delta_mean"].abs().max())
primary_stability = stability_summary.loc[stability_summary["k"] == 30]
max_abs_jaccard_delta = float(primary_stability["jaccard_delta_dedup_minus_original_raw"].abs().max())

takeaway = f'''
- The 9,450 HCFIM columns reduce to **{len(deduplicated_hcfim_columns):,} structurally unique features**; **{len(hcfim_columns)-len(deduplicated_hcfim_columns):,} columns ({100*(len(hcfim_columns)-len(deduplicated_hcfim_columns))/len(hcfim_columns):.1f}%)** are exact aliases implied by the definitions of H and FIM.
- The historical **875** tests equal 25 region-band cells × (7 H + 21 C + 7 transform-labeled FIM). Collapsing FIM to its three probability domains gives the mathematically defensible **775-test** family.
- For AD, BH discoveries were **{int(ad_fdr['raw_1575'])}**, **{int(ad_fdr['historical_875'])}**, and **{int(ad_fdr['corrected_775'])}** under the raw, 875-test, and corrected 775-test families; FTD retained **{int(ftd_fdr['corrected_775'])}** under the corrected family.
- In repeated matched-k validation at k=30, the largest absolute mean ROC-AUC change after deduplication was **{max_abs_auc_delta:.3f}**. The largest absolute change in mean raw-name Jaccard stability was **{max_abs_jaccard_delta:.3f}**.
- These are internal sensitivity results. They correct the feature/test inventory and quantify algorithmic sensitivity; they do not provide external validation.
'''
display(Markdown(takeaway))

summary_payload = {
    "raw_hcfim_columns": len(hcfim_columns),
    "structurally_unique_hcfim_columns": len(deduplicated_hcfim_columns),
    "redundant_aliases": len(hcfim_columns) - len(deduplicated_hcfim_columns),
    "empirically_unique_value_vectors": int(empirical_unique_vectors),
    "fdr_summary": fdr_summary.to_dict(orient="records"),
    "single_partition_delta": single_delta.to_dict(orient="records"),
    "paired_auc_summary": auc_summary.to_dict(orient="records"),
    "selection_stability_summary": stability_summary.to_dict(orient="records"),
}
(OUT / "audit_summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

output_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in OUT.rglob("*") if path.is_file())
audit_manifest = {
    "notebook": "12_feature_duplicate_count_fdr_stability_auc_audit.ipynb",
    "seed": SEED,
    "n_repeats": N_REPEATS,
    "outer_folds": OUTER_FOLDS,
    "k_values": K_VALUES,
    "input_feature_matrix": str(INPUT_FEATURES),
    "input_sha256": input_hash,
    "source_notebook_10_manifest": str(manifest_path),
    "outputs": output_files,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
(OUT / "notebook_12_manifest.json").write_text(json.dumps(audit_manifest, indent=2), encoding="utf-8")
print("Validation status: all assertions passed and the audit completed top-to-bottom.")
print(json.dumps(audit_manifest, indent=2))
